<a href="https://colab.research.google.com/github/DiegoMauricioSuaza/Actividad2/blob/main/Actividad_did%C3%A1ctica_2_M3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from datetime import datetime
import random

# Configurar semilla para reproducibilidad
np.random.seed(42)
random.seed(42)

# Parámetros del problema
HORAS_OPERACION = 8
MINUTOS_OPERACION = HORAS_OPERACION * 60  # 480 minutos
PORCENTAJE_RETIROS = 0.70
PORCENTAJE_PAGOS = 0.30
NUM_REPLICAS = 10

# Tabla 1: Tiempos de servicio y llegada (en minutos)
# Formato: {tipo_accion: {tipo_usuario: {'servicio': tiempo, 'llegada': tiempo}}}
tiempos = {
    'Retiro': {
        'Rápido': {'servicio': 1, 'llegada': 1},
        'Normal': {'servicio': 2, 'llegada': 2},
        'Lento': {'servicio': 3, 'llegada': 3},
        'Muy lento': {'servicio': 4, 'llegada': 3}
    },
    'Consignación': {
        'Rápido': {'servicio': 3, 'llegada': 1},
        'Normal': {'servicio': 3, 'llegada': 2},
        'Lento': {'servicio': 5, 'llegada': 3},
        'Muy lento': {'servicio': 7, 'llegada': 4}
    }
}

# Tabla 2: Probabilidades de tipos de usuario
probabilidades = {
    'Retiro': {
        'Rápido': 0.23,
        'Normal': 0.40,
        'Lento': 0.17,
        'Muy lento': 0.20
    },
    'Consignación': {
        'Rápido': 0.10,
        'Normal': 0.20,
        'Lento': 0.30,
        'Muy lento': 0.40
    }
}

class MM1Queue:
    """Clase para simular una cola M/M/1"""

    def __init__(self, cashier_id, action_type):
        self.cashier_id = cashier_id
        self.action_type = action_type  # 'Retiro' o 'Consignación'
        self.customers = []
        self.waiting_times = []
        self.service_times = []
        self.total_time_in_system = []
        self.user_types_count = {'Rápido': 0, 'Normal': 0, 'Lento': 0, 'Muy lento': 0}

    def generate_exponential(self, mean):
        """Genera tiempo exponencial con media dada"""
        return np.random.exponential(mean)

    def get_user_type(self):
        """Determina el tipo de usuario según probabilidades"""
        types = list(probabilidades[self.action_type].keys())
        probs = list(probabilidades[self.action_type].values())
        return np.random.choice(types, p=probs)

    def simulate_day(self):
        """Simula un día completo de operación"""
        current_time = 0
        server_free_time = 0
        customer_id = 0

        # Generar llegadas durante el día
        while current_time < MINUTOS_OPERACION:
            user_type = self.get_user_type()
            inter_arrival = self.generate_exponential(
                tiempos[self.action_type][user_type]['llegada']
            )
            current_time += inter_arrival

            if current_time >= MINUTOS_OPERACION:
                break

            customer_id += 1
            service_time = self.generate_exponential(
                tiempos[self.action_type][user_type]['servicio']
            )

            # Calcular tiempos
            arrival_time = current_time
            service_start = max(arrival_time, server_free_time)
            waiting_time = service_start - arrival_time
            departure_time = service_start + service_time

            self.customers.append({
                'id': customer_id,
                'arrival': arrival_time,
                'service_start': service_start,
                'departure': departure_time,
                'waiting_time': waiting_time,
                'service_time': service_time,
                'total_time': waiting_time + service_time,
                'user_type': user_type
            })

            self.waiting_times.append(waiting_time)
            self.service_times.append(service_time)
            self.total_time_in_system.append(waiting_time + service_time)
            self.user_types_count[user_type] += 1

            server_free_time = departure_time

        return self.get_statistics()

    def get_statistics(self):
        """Calcula estadísticas de la simulación"""
        if not self.customers:
            return None

        return {
            'cashier_id': self.cashier_id,
            'action_type': self.action_type,
            'total_customers': len(self.customers),
            'avg_waiting_time': np.mean(self.waiting_times),
            'avg_service_time': np.mean(self.service_times),
            'avg_total_time': np.mean(self.total_time_in_system),
            'max_waiting_time': max(self.waiting_times),
            'utilization': sum(self.service_times) / MINUTOS_OPERACION,
            'user_types': self.user_types_count.copy()
        }

def run_simulation(num_cashiers_withdrawal, num_cashiers_payment, replication):
    """Ejecuta una réplica de la simulación con configuración dada"""

    cashiers = []
    results = []

    # Crear cajeros de retiro
    for i in range(num_cashiers_withdrawal):
        cashier = MM1Queue(f"Retiro_{i+1}", 'Retiro')
        cashiers.append(cashier)

    # Crear cajeros de pago
    for i in range(num_cashiers_payment):
        cashier = MM1Queue(f"Pago_{i+1}", 'Consignación')
        cashiers.append(cashier)

    # Distribuir clientes entre cajeros del mismo tipo
    # Simular cada cajero independientemente pero con llegada proporcional
    for cashier in cashiers:
        stats = cashier.simulate_day()
        if stats:
            results.append(stats)

    return results

def analyze_configurations():
    """Analiza diferentes configuraciones de cajeros"""

    print("=" * 80)
    print("SIMULACIÓN DEL BANCO DE COLOMBIA - SISTEMA M/M/1")
    print("=" * 80)
    print(f"\nParámetros:")
    print(f"  - Horas de operación: {HORAS_OPERACION} horas ({MINUTOS_OPERACION} minutos)")
    print(f"  - Porcentaje retiros: {PORCENTAJE_RETIROS*100}%")
    print(f"  - Porcentaje pagos: {PORCENTAJE_PAGOS*100}%")
    print(f"  - Número de réplicas: {NUM_REPLICAS}")
    print("\n" + "=" * 80)

    # Configuraciones a probar
    configurations = [
        (1, 2, "1 Retiro + 2 Pagos"),
        (2, 1, "2 Retiros + 1 Pago")
    ]

    all_results = {}

    for num_withdrawal, num_payment, config_name in configurations:
        print(f"\n{'='*80}")
        print(f"CONFIGURACIÓN: {config_name}")
        print(f"{'='*80}\n")

        replication_results = []

        for rep in range(1, NUM_REPLICAS + 1):
            results = run_simulation(num_withdrawal, num_payment, rep)
            replication_results.append(results)

            print(f"Réplica {rep}:")
            for r in results:
                print(f"  {r['cashier_id']}: {r['total_customers']} clientes, "
                      f"Tiempo esp. prom: {r['avg_waiting_time']:.2f} min, "
                      f"Tiempo serv. prom: {r['avg_service_time']:.2f} min")
            print()

        all_results[config_name] = replication_results

        # Calcular estadísticas agregadas
        calculate_aggregate_statistics(replication_results, config_name)

    return all_results

def calculate_aggregate_statistics(replication_results, config_name):
    """Calcula estadísticas agregadas de todas las réplicas"""

    print(f"\n{'-'*80}")
    print(f"ESTADÍSTICAS AGREGADAS - {config_name}")
    print(f"{'-'*80}")

    # 1. Cajero con menor y mayor tiempo promedio de atención
    all_service_times = []
    cashier_stats = {}

    for rep_results in replication_results:
        for r in rep_results:
            cashier_id = r['cashier_id']
            if cashier_id not in cashier_stats:
                cashier_stats[cashier_id] = []
            cashier_stats[cashier_id].append(r['avg_service_time'])

    print("\n1. TIEMPOS PROMEDIO DE ATENCIÓN POR CAJERO:")
    avg_by_cashier = {}
    for cashier_id, times in cashier_stats.items():
        avg_time = np.mean(times)
        avg_by_cashier[cashier_id] = avg_time
        print(f"   {cashier_id}: {avg_time:.2f} minutos")

    min_cashier = min(avg_by_cashier, key=avg_by_cashier.get)
    max_cashier = max(avg_by_cashier, key=avg_by_cashier.get)

    print(f"\n   ✓ Cajero con MENOR tiempo: {min_cashier} ({avg_by_cashier[min_cashier]:.2f} min)")
    print(f"   ✓ Cajero con MAYOR tiempo: {max_cashier} ({avg_by_cashier[max_cashier]:.2f} min)")

    # 2. Promedio de usuarios de cada tipo en todos los cajeros
    print("\n2. PROMEDIO DE USUARIOS POR TIPO (todos los cajeros):")
    total_user_types = {'Rápido': 0, 'Normal': 0, 'Lento': 0, 'Muy lento': 0}

    for rep_results in replication_results:
        for r in rep_results:
            for user_type, count in r['user_types'].items():
                total_user_types[user_type] += count

    avg_user_types = {k: v/NUM_REPLICAS for k, v in total_user_types.items()}
    for user_type, avg in avg_user_types.items():
        print(f"   {user_type}: {avg:.1f} usuarios/día (promedio)")

    # 3. Total de usuarios por tipo en cada réplica
    print("\n3. TOTAL DE USUARIOS POR TIPO EN CADA RÉPLICA:")
    replica_totals = []

    for i, rep_results in enumerate(replication_results, 1):
        rep_total = {'Rápido': 0, 'Normal': 0, 'Lento': 0, 'Muy lento': 0}
        total_customers = 0

        for r in rep_results:
            for user_type, count in r['user_types'].items():
                rep_total[user_type] += count
            total_customers += r['total_customers']

        replica_totals.append((i, rep_total, total_customers))
        print(f"   Réplica {i}: Rápido={rep_total['Rápido']}, Normal={rep_total['Normal']}, "
              f"Lento={rep_total['Lento']}, Muy lento={rep_total['Muy lento']}, "
              f"Total={total_customers}")

    # Encontrar réplica con menor cantidad de usuarios
    min_replica = min(replica_totals, key=lambda x: x[2])
    print(f"\n   ✓ Modelo con MENOR cantidad de usuarios: Réplica {min_replica[0]} "
          f"({min_replica[2]} usuarios)")

    # 4. Análisis de tiempos de espera para decidir si se necesita nuevo cajero
    print("\n4. ANÁLISIS DE TIEMPOS DE ESPERA - ¿SE REQUIERE NUEVO CAJERO?")

    all_waiting_times = []
    all_utilizations = []

    for rep_results in replication_results:
        for r in rep_results:
            all_waiting_times.append(r['avg_waiting_time'])
            all_utilizations.append(r['utilization'])

    avg_waiting = np.mean(all_waiting_times)
    max_waiting = np.max(all_waiting_times)
    avg_utilization = np.mean(all_utilizations)

    print(f"   Tiempo promedio de espera general: {avg_waiting:.2f} minutos")
    print(f"   Máximo tiempo promedio de espera: {max_waiting:.2f} minutos")
    print(f"   Utilización promedio del sistema: {avg_utilization*100:.1f}%")

    # Criterios para decidir si se necesita nuevo cajero
    needs_new_cashier = False
    reasons = []

    if avg_waiting > 5:  # Si el tiempo promedio de espera es mayor a 5 minutos
        needs_new_cashier = True
        reasons.append("Tiempo promedio de espera > 5 minutos")

    if avg_utilization > 0.85:  # Si la utilización es mayor al 85%
        needs_new_cashier = True
        reasons.append("Utilización del sistema > 85%")

    if max_waiting > 15:  # Si algún cajero tiene espera promedio > 15 minutos
        needs_new_cashier = True
        reasons.append("Tiempo máximo de espera > 15 minutos")

    if needs_new_cashier:
        print(f"\n   ⚠ RECOMENDACIÓN: SE REQUIERE NUEVO CAJERO")
        for reason in reasons:
            print(f"      - {reason}")
    else:
        print(f"\n   ✓ RECOMENDACIÓN: NO se requiere cajero adicional")
        print("      Los tiempos de espera y utilización están dentro de límites aceptables")

    return {
        'avg_waiting_time': avg_waiting,
        'avg_utilization': avg_utilization,
        'needs_new_cashier': needs_new_cashier
    }

def compare_and_recommend(all_results):
    """Compara configuraciones y hace recomendaciones"""

    print("\n" + "="*80)
    print("COMPARACIÓN DE CONFIGURACIONES Y RECOMENDACIÓN FINAL")
    print("="*80)

    # Calcular métricas clave para cada configuración
    config_metrics = {}

    for config_name, replication_results in all_results.items():
        total_customers = 0
        total_waiting = 0
        total_service = 0
        count = 0

        for rep_results in replication_results:
            for r in rep_results:
                total_customers += r['total_customers']
                total_waiting += r['avg_waiting_time']
                total_service += r['avg_service_time']
                count += 1

        config_metrics[config_name] = {
            'avg_customers': total_customers / NUM_REPLICAS,
            'avg_waiting': total_waiting / count,
            'avg_service': total_service / count
        }

    print("\nMÉTRICAS POR CONFIGURACIÓN:")
    print("-" * 80)
    print(f"{'Configuración':<30} {'Clientes/día':<15} {'Esp. prom (min)':<18} {'Serv. prom (min)':<15}")
    print("-" * 80)

    for config, metrics in config_metrics.items():
        print(f"{config:<30} {metrics['avg_customers']:<15.1f} "
              f"{metrics['avg_waiting']:<18.2f} {metrics['avg_service']:<15.2f}")

    # Recomendación basada en el volumen de clientes
    print("\n" + "="*80)
    print("RECOMENDACIÓN FINAL")
    print("="*80)

    # La configuración que mejor atienda el 70% de retiros
    config_1_2 = config_metrics['1 Retiro + 2 Pagos']['avg_customers']
    config_2_1 = config_metrics['2 Retiros + 1 Pago']['avg_customers']

    print(f"\nDado que el 70% de los usuarios realiza RETIROS:")
    print(f"  - Configuración '1 Retiro + 2 Pagos': {config_1_2:.1f} clientes/día")
    print(f"  - Configuración '2 Retiros + 1 Pago': {config_2_1:.1f} clientes/día")

    if config_2_1 > config_1_2:
        print(f"\n✓ RECOMENDACIÓN: Asignar 2 cajeros para RETIROS y 1 cajero para PAGOS")
        print("  Esta configuración permite atender mejor el alto volumen de retiros (70%)")
    else:
        print(f"\n✓ RECOMENDACIÓN: Asignar 1 cajero para RETIROS y 2 cajeros para PAGOS")
        print("  Esta configuración muestra mejor desempeño general")

    print("\n" + "="*80)
    print("FIN DEL ANÁLISIS")
    print("="*80)

# Ejecutar la simulación
if __name__ == "__main__":
    results = analyze_configurations()
    compare_and_recommend(results)

SIMULACIÓN DEL BANCO DE COLOMBIA - SISTEMA M/M/1

Parámetros:
  - Horas de operación: 8 horas (480 minutos)
  - Porcentaje retiros: 70.0%
  - Porcentaje pagos: 30.0%
  - Número de réplicas: 10


CONFIGURACIÓN: 1 Retiro + 2 Pagos

Réplica 1:
  Retiro_1: 206 clientes, Tiempo esp. prom: 11.57 min, Tiempo serv. prom: 2.29 min
  Pago_1: 160 clientes, Tiempo esp. prom: 121.22 min, Tiempo serv. prom: 4.56 min
  Pago_2: 147 clientes, Tiempo esp. prom: 184.99 min, Tiempo serv. prom: 5.41 min

Réplica 2:
  Retiro_1: 218 clientes, Tiempo esp. prom: 67.77 min, Tiempo serv. prom: 2.45 min
  Pago_1: 170 clientes, Tiempo esp. prom: 211.49 min, Tiempo serv. prom: 5.26 min
  Pago_2: 166 clientes, Tiempo esp. prom: 235.32 min, Tiempo serv. prom: 5.36 min

Réplica 3:
  Retiro_1: 227 clientes, Tiempo esp. prom: 37.60 min, Tiempo serv. prom: 2.31 min
  Pago_1: 166 clientes, Tiempo esp. prom: 150.10 min, Tiempo serv. prom: 4.44 min
  Pago_2: 171 clientes, Tiempo esp. prom: 171.20 min, Tiempo serv. prom: 5.0